<a href="https://colab.research.google.com/github/Naildelyn/sistemas-opertativos-2026-2/blob/main/practica2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EJERCICIO 1

In [ ]:
import time
import sys
import os

def RoundRobin(Procesos, q):
    Cola = [{"nombre": P["nombre"],
             "restante": P["burst"]} for P in Procesos]

    tiempo = 0
    espera = {P["nombre"]: 0 for P in Procesos}


    while Cola:
        # Sacamos el primer proceso disponible
        actual = Cola.pop(0)

        # Calculamos cuánto tiempo ejecutará en este turno
        turno = min(q, actual["restante"])
        sobra = actual["restante"] - turno

        print(f"En el tiempo {tiempo}, el proceso {actual['nombre']} "
              f"correrá por {turno} ms, y le faltan {sobra} ms para terminar.")

        actual["restante"] = sobra
        tiempo += turno

        # Si todavía le falta tiempo, vuelve a la cola
        if actual["restante"] > 0:
            Cola.append(actual)
        else:
            print(f"El proceso {actual['nombre']} terminó en el tiempo {tiempo}.\n")


    if espera:
        promedio = sum(espera.values()) / len(espera)
        print(f"El tiempo promedio de espera de los procesos fue de {promedio} ms")
    else:
        print("No hay procesos para calcular el tiempo promedio de espera.")


Procesos = [
    {"nombre": "P1", "burst": 8},
    {"nombre": "P2", "burst": 3},
    {"nombre": "P3", "burst": 2}
]

RoundRobin(Procesos, q=2)

En el tiempo 0, el proceso P1 correrá por 2 ms, y le faltan 6 ms para terminar.
En el tiempo 2, el proceso P2 correrá por 2 ms, y le faltan 1 ms para terminar.
En el tiempo 4, el proceso P3 correrá por 2 ms, y le faltan 0 ms para terminar.
El proceso P3 terminó en el tiempo 6.

En el tiempo 6, el proceso P1 correrá por 2 ms, y le faltan 4 ms para terminar.
En el tiempo 8, el proceso P2 correrá por 1 ms, y le faltan 0 ms para terminar.
El proceso P2 terminó en el tiempo 9.

En el tiempo 9, el proceso P1 correrá por 2 ms, y le faltan 2 ms para terminar.
En el tiempo 11, el proceso P1 correrá por 2 ms, y le faltan 0 ms para terminar.
El proceso P1 terminó en el tiempo 13.

El tiempo promedio de espera de los procesos fue de 0.0 ms


EJERCICIO 2

In [ ]:
import multiprocessing
import time
import random

def RoundRobin(Procesos: list[dict], Quantum: int) -> None:
    """Simula Round Robin e imprime diagrama de Gantt y tiempo de espera por proceso.
    Args:
        Procesos: Lista de dicts con keys: nombre, burst.
        Quantum: Tiempo máximo de CPU por turno (ms).
    """
    Cola     = [{"nombre": P["nombre"], "restante": P["burst"]} for P in Procesos]
    Tiempo   = 0
    Espera   = {P["nombre"]: 0 for P in Procesos} #2
    Diagrama = []


    print(f"Quantum = {Quantum} ms\n{'-' * 40}")


    while Cola:
        Actual = Cola.pop(0)
        Turno  = min(Quantum, Actual["restante"])
        Sobra = Actual['restante'] - Turno
        print(f"t={Tiempo}ms — {Actual['nombre']} correrá  {Turno}ms    " f"(restante: {Actual['restante']} → {Sobra})")


        for Esperando in Cola:
            Espera[Esperando["nombre"]] += Turno


        Tiempo             += Turno
        Actual["restante"] -= Turno


        if Actual["restante"] > 0:
            Cola.append(Actual)
        else:
            print(f"  t={Tiempo}ms — {Actual['nombre']} TERMINÓ")


    Promedio = sum(Espera.values()) / len(Espera) #2


    print(f"\n{'=' * 40}\nProceso   Espera\n{'-' * 40}")
    for Nombre, TiempoEspera in Espera.items():
        print(f"  {Nombre}   {TiempoEspera} ms")
    print(f"{'-' * 40}\n  Promedio  {Promedio:.1f} ms")




if __name__ == "__main__":


    Procesos = [
        {"nombre": "P1", "burst": 8},
        {"nombre": "P2", "burst": 3},
        {"nombre": "P3", "burst": 2},
    ]


    print("ESCENARIO 1 — Quantum pequeño (q=2)")
    print("=" * 40)
    RoundRobin([{"nombre": P["nombre"], "burst": P["burst"]} for P in Procesos], Quantum=2)


    print("\nESCENARIO 2 — Quantum grande (q=10)")
    print("=" * 40)
    RoundRobin([{"nombre": P["nombre"], "burst": P["burst"]} for P in Procesos], Quantum=10)

ESCENARIO 1 — Quantum pequeño (q=2)
Quantum = 2 ms
----------------------------------------
t=0ms — P1 correrá  2ms    (restante: 8 → 6)
t=2ms — P2 correrá  2ms    (restante: 3 → 1)
t=4ms — P3 correrá  2ms    (restante: 2 → 0)
  t=6ms — P3 TERMINÓ
t=6ms — P1 correrá  2ms    (restante: 6 → 4)
t=8ms — P2 correrá  1ms    (restante: 1 → 0)
  t=9ms — P2 TERMINÓ
t=9ms — P1 correrá  2ms    (restante: 4 → 2)
t=11ms — P1 correrá  2ms    (restante: 2 → 0)
  t=13ms — P1 TERMINÓ

Proceso   Espera
----------------------------------------
  P1   5 ms
  P2   6 ms
  P3   4 ms
----------------------------------------
  Promedio  5.0 ms

ESCENARIO 2 — Quantum grande (q=10)
Quantum = 10 ms
----------------------------------------
t=0ms — P1 correrá  8ms    (restante: 8 → 0)
  t=8ms — P1 TERMINÓ
t=8ms — P2 correrá  3ms    (restante: 3 → 0)
  t=11ms — P2 TERMINÓ
t=11ms — P3 correrá  2ms    (restante: 2 → 0)
  t=13ms — P3 TERMINÓ

Proceso   Espera
----------------------------------------
  P1   0 ms
  P2   